# Piso 3ºB · Capítulo → audio (Qwen3-TTS, gratis)
**Cada día:** 1) pega el capítulo en la celda 3, 2) *Entorno de ejecución → Ejecutar todo*, 3) se descarga `audio_capitulo.json`, 4) en la web de Piso 3ºB cárgalo en «Audio de Colab» y pulsa «Crear vídeo».

Antes de la primera vez: *Entorno de ejecución → Cambiar tipo → GPU T4*, y pon en la celda 2 las semillas que elegiste.

In [ ]:
# 1) Instalar y cargar el modelo (tarda unos minutos solo la primera vez de cada sesión)
!pip install -q -U qwen-tts soundfile
import torch, numpy as np, random, json, re, base64
from qwen_tts import Qwen3TTSModel
model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")
def seed(n):
    random.seed(n); np.random.seed(n); torch.manual_seed(n); torch.cuda.manual_seed_all(n)

In [ ]:
# 2) Voces: semilla elegida de cada personaje (la de la prueba v2). yo = narrador, amigo = Dani, amiga = Marta
ELEGIDAS = {"yo": 1, "amigo": 1, "amiga": 1, "jefe": 1, "abuela": 1, "madre": 1}

ESP = ("Español de España, acento castellano de Madrid: distinción de 'z' y 'c' (como 'th'), 's' fuerte, "
       "jota fuerte, ritmo peninsular. Nada de acento latinoamericano ni neutro. ")
VOCES = {
 "yo":     ESP + "Hombre de unos 22 años, narrador cercano, irónico y cansado, tono medio.",
 "amigo":  ESP + "Hombre joven de unos 22 años, voz relajada y bromista, algo grave, muy coloquial.",
 "amiga":  ESP + "Mujer joven de unos 22 años, voz clara y enérgica, lista y sarcástica.",
 "jefe":   ESP + "Hombre de unos 55 años, voz ronca y autoritaria, casero gruñón.",
 "abuela": ESP + "Mujer de unos 75 años, voz temblorosa y cascada, vecina cotilla pero cariñosa.",
 "madre":  ESP + "Mujer de unos 50 años, voz cálida y algo agobiada, madre española que se preocupa por todo.",
}
# Para Emily (americana hablando español), cuando la tengamos en la serie:
# VOCES["emily"] = "Chica estadounidense de unos 20 años, de Massachusetts, que habla español con acento inglés americano muy marcado, voz alegre."
# ELEGIDAS["emily"] = 1

EMO = {  # estado de ánimo de la escena -> indicación de interpretación
 "neutro":   "Habla con naturalidad.",
 "cansado":  "Habla muy cansado, con voz lenta y apagada, suspirando.",
 "enfadado": "Habla enfadado, con voz fuerte, tensa y rápida.",
 "alegre":   "Habla alegre y emocionado, con energía y sonriendo.",
 "risa":     "Habla riéndose, divertido, con risa contenida.",
 "panico":   "Habla en pánico, muy rápido y agudo, casi gritando.",
 "duda":     "Habla dudando, con tono inseguro y pausas cortas.",
}
MOOD = {"feliz":"alegre","sonrisa":"alegre","risa":"risa","panic":"panico","panico":"panico","sorpresa":"panico","enfado":"enfadado",
        "triste":"cansado","dormido":"cansado","duda":"duda","pensando":"duda","asco":"enfadado","serio":"neutro","hablar":"neutro","neutro":"neutro"}

In [ ]:
# 3) PEGA AQUÍ EL CAPÍTULO de ChatGPT (entre las tres comillas). Vale con o sin el bloque ```json
CAPITULO = r'''

'''

In [ ]:
# 4) Generar el audio de cada escena
txt = CAPITULO.strip()
txt = txt[txt.index("{"): txt.rindex("}") + 1]
cap = json.loads(txt)
escenas = cap["escenas"][:40]; print(len(escenas), "escenas")
GAP = 0.12  # segundos entre trozos, igual que en la web

def limpiar(t): return re.sub(r"[*_#]", "", t).strip()
def trozos(e):
    t = re.sub(r"\s+", " ", e["texto"]).strip(); habla = e.get("habla") or "yo"; out = []
    for m in re.finditer(r"«([^»]*)»|([^«»]+)", t):
        if m.group(1) is not None:
            if m.group(1).strip(): out.append((habla, m.group(1).strip()))
        elif re.search(r"[\wÁÉÍÓÚáéíóúÑñ]", m.group(2)):
            out.append(("yo", re.sub(r"^[\s.,;:!?]+", "", m.group(2)).strip()))
    return out or [("yo", t)]
def animo(e, quien):
    for p in ((e.get("plano") or {}).get("pers") or []):
        if p.get("quien") == quien: return MOOD.get(p.get("mood"), "neutro")
    return "neutro"
def recortar(w, sr):
    a = np.abs(w); idx = np.where(a > 0.012)[0]
    if len(idx) == 0: return w
    return w[max(0, idx[0] - int(sr*.03)): min(len(w), idx[-1] + int(sr*.06))]

audios = []; SR = None
for i, e in enumerate(escenas):
    partes = []
    for quien, t in trozos(e):
        v = quien if quien in VOCES else "yo"
        seed(ELEGIDAS[v])
        wavs, sr = model.generate_voice_design(text=limpiar(t), language="Spanish",
                                               instruct=VOCES[v] + " " + EMO[animo(e, quien)])
        SR = sr; partes.append(recortar(np.asarray(wavs[0], dtype=np.float32), sr))
    gap = np.zeros(int(SR * GAP), dtype=np.float32); piezas = []
    for k, p in enumerate(partes):
        piezas.append(p); 
        if k < len(partes) - 1: piezas.append(gap)
    audios.append(np.concatenate(piezas)); print(f"escena {i+1}/{len(escenas)} · {len(audios[-1])/SR:.1f} s")

In [ ]:
# 5) Guardar y descargar audio_capitulo.json (súbelo en la web, en «Audio de Colab»)
def b64(w):
    pk = np.max(np.abs(w)) or 1.0; w = w * min(1.0, 0.95 / pk) if pk > 0.95 else w
    return base64.b64encode((np.clip(w, -1, 1) * 32767).astype("<i2").tobytes()).decode()
json.dump({"v": 1, "fs": int(SR), "escenas": [b64(w) for w in audios]}, open("audio_capitulo.json", "w"))
from google.colab import files; files.download("audio_capitulo.json")
print("Total:", round(sum(len(a) for a in audios) / SR), "s")